**Pachinko and the Central Limit Theorem.**
A pachinko machine is a physical demonstration of the **Central Limit Theorem**.
Each ball is dropped from the top and at every level bounces left or right
with equal probability, accumulating a random walk over `num_levels` steps.
The final slot position is the sum of `num_levels` independent $\pm 1$ coin flips,
divided by 2 to convert from step-sum to slot index.

By the CLT, this sum converges to a Gaussian (normal) distribution as
`num_levels` grows, regardless of the shape of the individual step distribution.
The standard deviation of the final positions is $\sigma = \sqrt{n}/2$
where $n$ is the number of levels.

The `@njit` decorator from Numba compiles the simulation to native machine
code, enabling millions of balls to be simulated in milliseconds.

In [ ]:
"""pachinko_normal.ipynb"""

# Cell 01 - Numba-accelerated pachinko simulation
# Each ball takes num_levels steps of +1 or -1 with equal probability.
# The final slot index is the accumulated displacement divided by 2.

import numpy as np
from numba import njit


@njit
def pachinko_normal(num_balls: int, num_levels: int) -> np.ndarray:
    np.random.seed(2016)
    balls = np.zeros(num_balls)
    for ball_num in range(num_balls):
        slot = 0
        for _ in range(num_levels):
            slot += -1 if np.random.rand() < 0.5 else 1
        balls[ball_num] = slot // 2  # convert step-sum to slot index
    return balls


# Warm-up call to trigger JIT compilation of the function
sample = pachinko_normal(num_balls=5, num_levels=10)
print(f"Sample slot positions (5 balls, 10 levels): {sample}")

**Comparing the simulation to the analytic normal distribution.**
`run_simulation` drops `total_balls` through `total_levels` levels,
bins the final slot positions into a probability distribution, and
overlays the exact Gaussian PDF with the same mean and standard deviation.
As the number of balls and levels increases, the two curves converge.

In [ ]:
# Cell 02 - Simulation runner and plot function

import matplotlib.pyplot as plt
from scipy.stats import norm


def run_simulation(total_balls: int, total_levels: int) -> None:
    balls = pachinko_normal(total_balls, total_levels)

    # Bin balls into slots using vectorized np.bincount
    first_slot = total_levels // 2
    slot_indices = (first_slot + balls).astype(int)
    slots = np.bincount(slot_indices, minlength=total_levels + 1) / total_balls

    # Slot positions centered at zero
    x = np.linspace(-total_levels // 2, total_levels // 2, total_levels + 1)

    # Analytic normal PDF with matching mean and standard deviation
    mu = np.mean(balls)
    sigma = np.std(balls)
    norm_x = np.linspace(-total_levels // 2, total_levels // 2, 100)
    norm_y = norm(mu, sigma).pdf(norm_x)

    plt.plot(x, slots, color="blue", linewidth=2, label="Pachinko PDF")
    plt.plot(norm_x, norm_y, color="red", linewidth=2, label="Normal PDF")
    plt.title(
        f"Pachinko vs. Normal PDF ({total_balls:,} balls, {total_levels:,} levels)"
    )
    plt.xlabel("Slot Number")
    plt.ylabel("Probability")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


# Baseline: small ball count shows noisy, coarse approximation
run_simulation(total_balls=1_000, total_levels=10)

In [ ]:
# Cell 03 - 10x more balls: distribution smooths out noticeably

run_simulation(total_balls=10_000, total_levels=10)

In [ ]:
# Cell 04 - More levels: wider, smoother distribution (sigma grows as sqrt(levels))

run_simulation(total_balls=100_000, total_levels=20)

In [ ]:
# Cell 05 - 1M balls, 40 levels: near-perfect agreement with normal PDF

run_simulation(total_balls=1_000_000, total_levels=40)

In [ ]:
# Cell 06 - 1M balls, 80 levels: even wider Gaussian, visually indistinguishable
# from the analytic PDF - CLT convergence is complete at this scale

run_simulation(total_balls=1_000_000, total_levels=80)